# Phase 3 — Data Preparation

## 3.1 Define the Target Data Model & Cleaning Rules

We will establish:

1. Target transaction grain
2. Final column names
3. Final data types
4. Duplicate policy
5. Missing Customer ID policy
6. Missing Description policy
7. Country standardization policy
8. Transaction classification
9. Return/cancellation treatment
10. Non-merchandise treatment
11. Zero-price treatment
12. Negative-price treatment
13. Outlier treatment
14. Derived fields
15. Rules for the reporting dataset

### 3.1.1 Target Transaction Grain

The cleaned transaction dataset will retain the transaction-line grain of the
source data.

Each row represents one transaction line recorded in the original e-commerce
export.

An `Invoice` is therefore not treated as a unique row identifier because a
single invoice can contain multiple transaction lines.

This grain preserves the detail required for subsequent invoice-level,
product-level, customer-level, country-level, and monthly analysis.

### 3.1.2 Target Schema and Standardized Column Names

The cleaned transaction dataset will use a consistent lowercase `snake_case`
naming convention to improve readability and compatibility across Python,
SQL, Excel, and reporting workflows.

| Source Field  | Target Field   | Expected Type       | Description                               |
| ------------- | -------------- | ------------------- | ----------------------------------------- |
| `Invoice`     | `invoice`      | string              | Invoice/transaction identifier            |
| `StockCode`   | `stock_code`   | string              | Product or transaction code               |
| `Description` | `description`  | string              | Product or transaction description        |
| `Quantity`    | `quantity`     | integer             | Quantity recorded on the transaction line |
| `InvoiceDate` | `invoice_date` | datetime            | Date and time associated with the invoice |
| `Price`       | `unit_price`   | numeric             | Price recorded for the transaction line   |
| `Customer ID` | `customer_id`  | nullable identifier | Customer identifier where available       |
| `Country`     | `country`      | string              | Country associated with the transaction   |

`invoice` and `stock_code` will remain string-based identifiers because the
source data contains both numeric and alphanumeric values.

`customer_id` will be treated as an identifier rather than a numerical
measurement. Missing customer identifiers will be preserved until the
appropriate business rule is established.

Derived fields such as transaction classification, line value, reporting
value, and time-based reporting fields will be added later after their
business definitions have been established.


### 3.1.3 Duplicate-Record Policy

Duplicate handling will distinguish between repeated invoice identifiers and
actual duplicated transaction lines.

A repeated `invoice` value is not considered a duplicate because the source
data is recorded at transaction-line level and a single invoice may contain
multiple legitimate transaction lines.

Exact duplicate transaction rows are treated as potential duplicated records.
After the relevant fields have been standardized, identical transaction lines
will be evaluated using the duplicate rule defined for the cleaning pipeline.

For the cleaned analytical dataset, one instance of an exact duplicate
transaction line will be retained and additional identical instances will be
removed.

The pipeline will maintain an audit trail containing the raw row count,
duplicate rows identified, duplicate rows removed, and resulting cleaned row
count.

No duplicate handling is based solely on the `invoice` field.


### 3.1.4 Missing Customer ID Policy

`Customer ID` contains substantial missingness in the source data, with
20.54% of records missing the field in the 2009–2010 reporting period and
24.93% missing in the 2010–2011 reporting period.

Missing `customer_id` values will not be used as an automatic reason for
removing transaction records.

Transaction records with missing customer identifiers will be preserved in
the cleaned transaction dataset when they satisfy the other applicable data
quality and transaction-classification rules.

Missing customer identifiers will remain explicitly missing. Customer IDs
will not be invented, forward-filled, replaced with arbitrary placeholder
identifiers, or inferred from surrounding transactions.

Records with valid customer identifiers may be used for customer-level
analysis. Records without a valid customer identifier cannot be reliably
attributed to an individual customer and will therefore be excluded from
customer-specific attribution while remaining available for other applicable
transaction, product, time, or geographic analysis.

This policy preserves potentially valid business transactions while ensuring
that customer-level analysis does not imply information that is not supported
by the source data.


### 3.1.5 Missing Description Policy

The `Description` field contains a relatively small amount of missing data:
2,928 records (0.56%) in the 2009–2010 reporting period and 1,454 records
(0.27%) in the 2010–2011 reporting period.

Missing `description` values will not be used as an automatic reason for
removing transaction records.

The transaction will be preserved when it satisfies the other applicable data
quality and transaction-classification rules. The missing description will
remain explicitly missing rather than being replaced with an invented product
description.

Where appropriate, later stages may investigate whether a reliable
description can be associated with a transaction using other validated fields,
such as `stock_code` or an established transaction classification. Any such
standardization will be based on documented rules rather than assumptions.

The `description` field will therefore be treated as a descriptive attribute,
while `stock_code` remains the primary product/transaction code available for
product-level identification.


### 3.1.6 Country Standardization Policy

The `Country` field contains no missing values in either reporting period.
However, the source contains country labels that may use different naming
conventions, including values such as `UK`, `EIRE`, `RSA`, and `Unspecified`.

Country values will therefore not be considered invalid solely because their
labels do not follow a single naming convention.

The initial cleaning process will preserve the source country values while
country-label standardization is defined.

Any later normalization will use an explicit and documented mapping based on
the reporting convention selected for the project. Country labels will not be
arbitrarily renamed, merged, or inferred.

`Unspecified` will remain distinct from a null or missing country value because
it is an explicit value present in the source data.

The objective is to produce consistent country-level reporting without
silently changing the geographic meaning of the source data.


### 3.1.7 Transaction Classification Policy

The cleaned transaction dataset will include a `transaction_type` field to
distinguish different types of activity represented in the source data.

The initial transaction taxonomy will include:

| Classification | Meaning                                                                  |
| -------------- | ------------------------------------------------------------------------ |
| `sale`         | Ordinary merchandise transaction                                         |
| `return`       | Negative merchandise transaction representing returned or reversed goods |
| `cancellation` | Cancellation-style transaction                                           |
| `adjustment`   | Financial or accounting adjustment                                       |
| `fee`          | Fee or charge transaction                                                |
| `shipping`     | Postage, carriage, or shipping-related transaction                       |
| `discount`     | Discount-related transaction                                             |
| `manual`       | Manual transaction                                                       |
| `other`        | Other non-standard transaction requiring separate treatment              |
| `unknown`      | Transaction that cannot be reliably classified using established rules   |

Transaction classification will not be based on a single field. The rules will
consider relevant combinations of `invoice`, `stock_code`, `description`,
`quantity`, `unit_price`, and `customer_id`.

The presence of a transaction classification does not by itself determine
whether a record is included in a particular business metric. Classification
and reporting inclusion will be treated as separate decisions.

Records that cannot be confidently classified will not be forced into an
arbitrary category. They will remain identifiable for further review.

Detailed classification rules for cancellations, returns, non-merchandise
transactions, zero-price records, and negative-price records will be defined
in subsequent Data Preparation steps.


### 3.1.8 Cancellation and Return Treatment Policy

Cancellation and return transactions will be retained in the cleaned transaction
dataset rather than removed during data preparation.

Invoice values beginning with `C` will be treated as a strong indicator of a
cancellation transaction, subject to the broader transaction-classification
rules.

Negative-quantity merchandise transactions that do not meet the cancellation
criteria may be classified as `return` transactions.

Negative quantity alone will not be used as a universal definition of a
cancellation because the Data Understanding phase identified negative-quantity
records that do not have cancellation-style invoice identifiers.

Cancellation and return records will therefore remain available for analysis,
quality monitoring, and reporting.

The reporting layer will distinguish between ordinary merchandise activity and
reversing activity so that gross sales, returns/cancellations, and net
merchandise sales can be calculated separately.

Negative quantities and negative line values will not be removed solely because
they are negative.

Unusual combinations of invoice prefix, quantity, price, or other transaction
attributes will be retained and identified for review rather than silently
deleted.

The objective is to preserve the commercial meaning of reversals while
preventing cancellations and returns from being incorrectly treated as
ordinary sales.


### 3.1.9 Non-Merchandise Transaction Treatment Policy

The source data contains transaction lines that do not represent ordinary
merchandise sales. Examples identified during Data Understanding include
postage, carriage, discounts, fees, bank charges, Amazon fees, manual
transactions, samples, bad-debt adjustments, test records, and gift vouchers.

These transactions will not be removed solely because they are non-merchandise.

They will remain in the cleaned transaction dataset where they satisfy the
overall data-quality rules and will be assigned an appropriate
`transaction_type`.

Known non-merchandise activity will be classified using relevant combinations
of `stock_code`, `description`, `invoice`, `quantity`, and `unit_price`.

The following categories will be used as part of the classification framework:

* `shipping` — postage, carriage, and related shipping activity
* `discount` — discount-related transactions
* `fee` — fees and charges such as Amazon fees or bank charges
* `adjustment` — financial or accounting adjustments
* `manual` — manually recorded transactions
* `other` — other identified non-merchandise activity
* `unknown` — activity that cannot yet be classified reliably

Alphabetic or non-numeric `stock_code` values will not automatically be treated
as non-merchandise because the Data Understanding phase identified legitimate
merchandise codes that also contain alphabetic characters.

Non-merchandise transactions will therefore be preserved for auditability and
potential operational analysis while being excluded from ordinary merchandise
sales metrics unless a specific reporting rule states otherwise.

The objective is to distinguish transaction classification from transaction
deletion and prevent operational, financial, or other non-merchandise activity
from being incorrectly reported as ordinary product sales.


### 3.1.10 Zero-Price and Negative-Price Treatment Policy

The source data contains both zero-price and negative-price transaction lines.

The Data Understanding phase identified 6,202 zero-price rows and 5 negative-price
rows across the two reporting periods.

Zero-price transactions will not be removed automatically. The presence of a
zero `unit_price` does not by itself establish that a transaction is invalid,
because zero-price records may represent different types of business activity.

Zero-price records will be retained and flagged for further classification and
reporting treatment. Their transaction type will be determined using the
broader classification rules rather than the price value alone.

Negative-price transactions will also be retained. The five negative-price
records identified during Data Understanding were associated with
`Adjust bad debt` transactions and will therefore be treated as financial
adjustments rather than ordinary merchandise sales.

The cleaning process will use derived quality indicators such as
`is_zero_price` and `is_negative_price` to make these conditions visible
without altering the original transaction values.

Negative prices associated with ordinary merchandise activity will be retained
but flagged for review rather than silently deleted.

Price anomalies will therefore be treated as classification and data-quality
signals rather than automatic deletion criteria.

The final reporting layer will determine whether these transactions contribute
to merchandise sales, returns, fees, adjustments, or other business metrics.
`Quantity × unit_price` will not be treated as revenue for every transaction
without considering its transaction classification.


### 3.1.11 Quantity and Numeric Validation Rules

The `quantity` and `unit_price` fields are core transaction measures and will
be explicitly validated before being used in derived calculations and business
reporting.

The target `quantity` field will be represented as an integer because the
source records transaction quantities as whole units.

Missing quantities will be treated as data-quality failures. Missing values
will not be replaced with zero or another inferred value.

A quantity of zero will be treated as an invalid transaction quantity because
it represents no recorded unit movement. The current source contains no zero
quantities, but the rule is retained for future recurring exports.

Negative quantities will be preserved because they may represent returns,
cancellations, or other transaction reversals. Negative quantities will not be
converted to positive values and will be interpreted using the transaction
classification rules.

The `unit_price` field must contain a numeric value for a transaction to pass
price completeness validation. Missing prices will not be replaced with zero
or another assumed value.

Zero unit prices will be retained and flagged because they may represent
legitimate business activity such as free items, samples, or other
non-standard transactions.

Negative unit prices will also be retained and flagged. Their treatment will
depend on the transaction context. For example, the negative-price
`Adjust bad debt` records identified during Data Understanding will be treated
as financial adjustments rather than ordinary merchandise sales.

Extremely large quantities or prices will not be removed solely because they
appear unusual. They will be flagged for investigation so that legitimate
large transactions are not incorrectly discarded.

A derived `line_value` field may be calculated as:

`line_value = quantity × unit_price`

`line_value` represents the mathematical value of the transaction line and
will not automatically be interpreted as revenue. Revenue and other business
metrics will be calculated using the established transaction classifications
and reporting rules.

The objective is to distinguish structural numeric validity from business
meaning while preserving unusual but potentially legitimate transactions.


### 3.1.12 Date and Time Standardization Rules

The source `InvoiceDate` field will be standardized to the target column name
`invoice_date` while preserving its datetime representation.

The complete date and time value will be retained. The cleaning process will
not truncate timestamps to date-only values because the time component may
support future operational or temporal analysis.

No timezone conversion will be applied because the source data does not
establish a timezone transformation requirement. The source datetime values
will therefore be preserved as provided.

The pipeline will validate that `invoice_date` contains valid datetime values.
Missing or unparseable dates will be treated as data-quality failures rather
than being replaced with inferred values.

Date ranges will also be monitored. Dates outside the documented temporal
coverage of the source will be flagged for investigation rather than
automatically deleted.

A derived `reporting_month` field will be created for monthly reporting.
This field will represent the reporting period associated with each
transaction while `invoice_date` continues to preserve the original
transaction timestamp.

The source workbook's sheet boundaries will not be used as a duplicate rule.
The presence of December 2010 in both source sheets does not by itself
indicate duplicate transactions. Duplicate handling will continue to follow
the transaction-level policy established in Section 3.1.3.

Additional date dimensions such as year, quarter, month name, or day of week
may be derived later when they support a defined analytical or reporting
requirement.

The objective is to preserve the original temporal information while creating
consistent time dimensions for automated business reporting.


### 3.1.13 Derived Fields and Reporting Metrics

Derived fields will be created to provide transaction-level classification,
quality indicators, and reporting dimensions without altering the original
business meaning of the source fields.

The initial transaction-level derived fields will include:

| Field               | Definition                                                                  |
| ------------------- | --------------------------------------------------------------------------- |
| `line_value`        | `quantity × unit_price`; mathematical transaction-line value                |
| `transaction_type`  | Business classification established by the transaction-classification rules |
| `reporting_month`   | Monthly reporting period derived from `invoice_date`                        |
| `is_zero_price`     | Indicates whether `unit_price` equals zero                                  |
| `is_negative_price` | Indicates whether `unit_price` is negative                                  |
| `is_return`         | Indicates that `transaction_type` is classified as `return`                 |
| `is_cancellation`   | Indicates that `transaction_type` is classified as `cancellation`           |

Derived fields will only be created when they provide clear analytical,
reporting, validation, or audit value.

`line_value` will represent the mathematical value of a transaction line and
will not automatically be interpreted as revenue. Business financial metrics
will use transaction classifications and explicit reporting inclusion rules.

The initial reporting KPI framework will include:

| KPI                     | Definition                                                |
| ----------------------- | --------------------------------------------------------- |
| Gross Merchandise Sales | Value of qualifying positive merchandise sales            |
| Returns / Cancellations | Value associated with qualifying merchandise reversals    |
| Net Merchandise Sales   | Gross Merchandise Sales less returns/cancellations        |
| Orders                  | Unique qualifying invoice identifiers                     |
| Units Sold              | Positive quantity from qualifying merchandise sales       |
| Units Returned          | Quantity associated with qualifying merchandise reversals |
| Unique Customers        | Distinct non-null `customer_id` values                    |
| Average Order Value     | Net Merchandise Sales divided by qualifying orders        |
| Return Rate             | Units Returned divided by Units Sold                      |
| Revenue Growth          | Period-over-period change in Net Merchandise Sales        |

Orders will be calculated from unique qualifying invoice identifiers rather
than transaction-row counts because the source data is stored at
transaction-line grain.

Unique customer counts will use only non-null `customer_id` values. Missing
customer identifiers will not be replaced or inferred.

Operational and non-merchandise activity such as shipping, fees, discounts,
manual transactions, and financial adjustments will remain available in the
cleaned dataset but will not automatically contribute to ordinary merchandise
sales KPIs.

The terminology `Net Merchandise Sales` will be preferred over an unrestricted
`Revenue` metric because the source contains multiple transaction types that
do not represent ordinary merchandise sales.

The final reporting layer will apply these definitions consistently across
monthly, product, country, and customer-level outputs.
